<div class="nb-header">
  <span class="nb-header__type">Tutorial</span>
  <h1 class="nb-header__title">Complete Workflows</h1>
  <p class="nb-header__subtitle">End-to-end patterns for common graph analytics tasks</p>
  <div class="nb-header__meta">
    <span class="nb-header__meta-item nb-header__meta-item--duration">45 min</span>
    <span class="nb-header__meta-item nb-header__meta-item--level">
      <span class="nb-difficulty nb-difficulty--advanced">
        <span class="nb-difficulty__dot"></span>
        <span class="nb-difficulty__dot"></span>
        <span class="nb-difficulty__dot"></span>
      </span>
      Advanced
    </span>
  </div>
  <div class="nb-header__tags"><span class="nb-header__tag">Workflow</span><span class="nb-header__tag">Patterns</span><span class="nb-header__tag">Best Practices</span></div>
</div>

<div class="nb-objectives">
  <h3 class="nb-objectives__title">What You'll Learn</h3>
  <ul class="nb-objectives__list">
    <li><strong>ETL Pattern</strong> - Extract, transform, load graph data</li>
    <li><strong>Analysis Pattern</strong> - Run algorithms and export results</li>
    <li><strong>Refresh Pattern</strong> - Update graphs with new data</li>
    <li><strong>Multi-tenant Pattern</strong> - Manage graphs for multiple users</li>
  </ul>
</div>

<div class="nb-section">
  <span class="nb-section__number">1</span>
  <div>
    <h2 class="nb-section__title">ETL Workflow</h2>
    <p class="nb-section__description">Warehouse to graph pipeline</p>
  </div>
</div>

In [ ]:
# Complete ETL workflow
from graph_olap import GraphOLAPClient

client = GraphOLAPClient.from_env()

# 1. Discover schema
tables = client.schemas.list_tables("analytics")

# 2. Create mapping
mapping = client.mappings.create(
    name="CustomerAnalytics",
    node_definitions=[...],
    edge_definitions=[...]
)

# 3. Create snapshot
snapshot = client.snapshots.create(mapping_id=mapping.id)
snapshot = client.snapshots.wait_for_ready(snapshot.id)

# 4. Deploy instance
instance = client.instances.create(snapshot_id=snapshot.id)
instance = client.instances.wait_for_running(instance.id)

# 5. Connect and query
conn = client.instances.connect(instance.id)

<div class="nb-section">
  <span class="nb-section__number">2</span>
  <div>
    <h2 class="nb-section__title">Analysis Workflow</h2>
    <p class="nb-section__description">Algorithm execution and export</p>
  </div>
</div>

In [ ]:
# Analysis workflow
conn = client.instances.connect(instance_id)

# Run algorithms
conn.algorithms.pagerank(label="Customer", write_property="influence")
conn.algorithms.louvain(label="Customer", write_property="segment")

# Export results
df = conn.query_df("""
    MATCH (c:Customer)
    RETURN c.id, c.name, c.influence, c.segment
""")

df.to_csv("customer_segments.csv", index=False)

<div class="nb-section">
  <span class="nb-section__number">3</span>
  <div>
    <h2 class="nb-section__title">Data Refresh Workflow</h2>
    <p class="nb-section__description">Update graphs with new data</p>
  </div>
</div>

In [ ]:
# Refresh workflow - new data, same schema

# Create new snapshot from existing mapping
new_snapshot = client.snapshots.create(
    mapping_id=mapping.id,
    description="Weekly refresh - 2024-01-15"
)
new_snapshot = client.snapshots.wait_for_ready(new_snapshot.id)

# Create new instance
new_instance = client.instances.create(snapshot_id=new_snapshot.id)

# Terminate old instance
client.instances.terminate(old_instance.id)

<div class="nb-section">
  <span class="nb-section__number">4</span>
  <div>
    <h2 class="nb-section__title">Production Patterns</h2>
    <p class="nb-section__description">Best practices for production</p>
  </div>
</div>

In [ ]:
from graph_olap import notebook
from graph_olap.testing import TestPersona

# Production pattern with automatic cleanup
ctx = notebook.test("ProductionWorkflow", persona=TestPersona.ANALYST_ALICE)

try:
    mapping = ctx.mapping(name="ProdGraph", ...)
    snapshot = ctx.snapshot(mapping)
    instance = ctx.instance(snapshot)
    
    conn = ctx.client.instances.connect(instance.id)
    # ... do work ...
    
finally:
    ctx.cleanup()  # Automatic cleanup

<div class="nb-takeaways">
  <h3 class="nb-takeaways__title">Key Takeaways</h3>
  <ul class="nb-takeaways__list">
    <li>ETL: Catalog discovery → Mapping → Snapshot → Instance</li>
    <li>Analysis: Connect → Algorithms → Query → Export</li>
    <li>Refresh: New snapshot from same mapping preserves schema</li>
    <li>Production: Use test context, handle errors, clean up resources</li>
  </ul>
</div>